# Class 6: From API Calls to Agents — First Principles

We use **OpenRouter** — one API key, hundreds of models, free tier available.

---

## Part 1: Setup & First API Call

In [ ]:
!uv pip install openai requests -q

### Setup OpenRouter

OpenRouter is **OpenAI-compatible** — same SDK, different `base_url`.

**To get your API key:**
1. Go to [openrouter.ai](https://openrouter.ai) → create a free account
2. Go to [openrouter.ai/keys](https://openrouter.ai/keys) → create a new key
3. Add it to Colab Secrets (🔑 icon in sidebar) as `OPENROUTER_API_KEY`

Free models available — no credit card needed.

In [ ]:
import os
import json
import re
import time
import requests
from openai import OpenAI
from getpass import getpass

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    api_key = getpass("Enter your OpenRouter API key: ")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is required.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

### Your first API call

This is **inference**. You send tokens, get tokens back. The model you trained in Class 5 does the same thing — the API just wraps it with HTTP.

In [ ]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{
        "role": "user",
        "content": "What is 2 + 2? Answer in one word."
    }],
    temperature=0.0,
    max_tokens=0.0,
)

print("Response:", response.choices[0].message.content)
print(f"Tokens - in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

### Temperature — quick demo

You know why this works: `softmax(logits / T)`
- T=0: deterministic
- T=1: variety
- T=2: chaos

In [ ]:
prompt = "Write a one-sentence story about a robot."

for temp in [0.0, 1.0, 2.0]:
    results = []

    for _ in range(3):
        r = client.chat.completions.create(
            model=FREE_MODEL,
            messages=[{
                "role": "user",
                "content": prompt
            }],
            temperature=temp,
            max_tokens=60,
        )

        content = r.choices[0].message.content

        if content:
            results.append(content.strip())
        else:
            results.append("[No content returned]")

    print(f"\nTemperature = {temp}:")
    for i, text in enumerate(results):
        print(f"   [{i + 1}] {text[:120]}")

### The model has NO memory

Multi-turn: the **entire conversation** is re-sent every time. "Chat memory" is your application re-sending history.

In [ ]:
conversation = [
    {
        "role": "system",
        "content": "You are a math tutor. Be concise"
    },
    {
        "role": "user",
        "content": "What is a derivative"
    }
]

r1 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    max_tokens=150
)

turn1 = r1.choices[0].message.content or "[No content returned]"
print("Turn 1:", turn1[:200])
print(f"  Prompt tokens: {r1.usage.prompt_tokens}")

conversation.append({"role": "assistant", "content": turn1})
conversation.append({"role": "user", "content": "Give me an example with x²."})

r2 = client.chat.completions.create(
    model=FREE_MODEL,
    messages=conversation,
    max_tokens=150
)

turn2 = r2.choices[0].message.content or "[No content returned]"
print(f"\nTurn 2: {turn2[:200]}")
print(f"  Prompt tokens: {r2.usage.prompt_tokens}")

print(f"\n→ Prompt tokens grew from {r1.usage.prompt_tokens} to {r2.usage.prompt_tokens}")

---
## Part 2: Function Calling — The Bridge to Agents

Function calling = the model can **REQUEST** tools. It outputs structured JSON saying "I want to call X with args Y." **YOUR** code executes the function and feeds the result back.

The model never "calls" anything. It generates text that says what it wants.

### Define tools

We describe tools so the model knows what's available. The model doesn't HAVE these tools — it can only REQUEST them.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city. Returns temperature and conditions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Tokyo'"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '(5 + 3) * 2'"}
                },
                "required": ["expression"]
            }
        }
    }
]

### The model decides to use a tool

In [ ]:
r = client.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=tools,
    temperature=0,
)

msg = r.choices[0].message
print(f"Finish reason: {r.choices[0].finish_reason}")
print(f"Content: {msg.content}")
print(f"Tool calls: {msg.tool_calls}")

if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"\n→ Model wants to call: {tc.function.name}({tc.function.arguments})")
    print("  It GENERATED JSON requesting a function. Our code must execute it.")

### Complete the tool call loop

`User → Model requests tool → OUR code runs tool → Result back to model → Final answer`

In [ ]:
# Fake tool implementations
def get_weather(city):
    fake = {
        "Tokyo": {"temp": "22°C", "condition": "partly cloudy"},
        "London": {"temp": "14°C", "condition": "rainy"},
        "Delhi": {"temp": "38°C", "condition": "sunny"},
    }
    return json.dumps(fake.get(city, {"temp": "unknown", "condition": "unknown"}))

def calculate(expression):
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": "Invalid characters"})
        return json.dumps({"result": eval(expression)})
    except Exception as e:
        return json.dumps({"error": str(e)})

TOOL_FNS = {"get_weather": get_weather, "calculate": calculate}

# Execute the tool call
if msg.tool_calls:
    tc = msg.tool_calls[0]
    fn = TOOL_FNS[tc.function.name]
    result = fn(**json.loads(tc.function.arguments))
    print(f"Tool result: {result}")

    # Feed result back to model
    messages = [
        {"role": "user", "content": "What's the weather in Tokyo?"},
        msg,
        {"role": "tool", "tool_call_id": tc.id, "content": result}
    ]

    r2 = client.chat.completions.create(
        model=TOOL_MODEL, messages=messages, tools=tools
    )
    print(f"\nFinal answer: {r2.choices[0].message.content}")
    print("\n→ Full cycle: user → model requests tool → we run it → model answers")

---
## Part 3: Build a ReAct Agent From Scratch

**No LangChain. No CrewAI. No frameworks.**

Just Python + an API + a loop.

The **ReAct** pattern: `THINK → ACT → OBSERVE → repeat until done`

The agent's system prompt
This prompt IS the architecture. It tells the model to alternate between THOUGHT and ACTION in a parseable format.

In [ ]:
REACT_SYSTEM_PROMPT = """You are a helpful assistant that solves problems step by step using tools.

You have access to these tools:
{tool_descriptions}

## How to respond

When you need to use a tool, respond in EXACTLY this format:

THOUGHT: <your reasoning about what to do next>
ACTION: <tool_name>
ACTION_INPUT: <arguments as valid JSON>

When you have enough information for the final answer:

THOUGHT: <your final reasoning>
FINAL_ANSWER: <your complete answer to the user>

## Rules
- Always start with THOUGHT
- Use only ONE action per turn
- Wait for the OBSERVATION before continuing
- If a tool returns an error, reason about it and try a different approach
- Be concise in your thoughts
"""

### Real tools

Wikipedia search hits a real API. The calculator does real math. These are the "hands" of our agent.

In [ ]:
def search_wikipedia(query):
    """Search Wikipedia and return a summary."""
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return json.dumps({
                "title": data.get("title", ""),
                "summary": data.get("extract", "No summary found.")[:800]
            })
        return json.dumps({"error": f"Page not found for '{query}'. Try a different term."})
    except Exception as e:
        return json.dumps({"error": str(e)})

def calculate_math(expression):
    """Safely evaluate a math expression."""
    try:
        allowed = set("0123456789+-*/.() eE")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": f"Invalid characters in: {expression}"})
        result = eval(expression)
        return json.dumps({"expression": expression, "result": round(result, 6)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def get_current_date():
    """Get the current date and time."""
    from datetime import datetime
    return json.dumps({"datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

# Tool registry
TOOLS = {
    "search_wikipedia": {
        "fn": search_wikipedia,
        "desc": "search_wikipedia(query: str) — Search Wikipedia. Use simple topic names like 'France' or 'Albert Einstein'."
    },
    "calculate": {
        "fn": calculate_math,
        "desc": "calculate(expression: str) — Evaluate a math expression. Example: '(5 + 3) * 2.5'"
    },
    "get_current_date": {
        "fn": get_current_date,
        "desc": "get_current_date() — Get today's date and time. Pass empty JSON: {}"
    },
}

print(f"Tools registered: {list(TOOLS.keys())}")

### 🔥 THE AGENT LOOP

This is the heart of the class. **~60 lines** that implement the same pattern as Claude Code and Codex.

Walk through it:
1. Build system prompt with tool descriptions
2. Enter a loop (max_iterations prevents infinite loops)
3. Each iteration: call the LLM, get text with THOUGHT + ACTION or FINAL_ANSWER
4. If FINAL_ANSWER → done
5. If ACTION → parse tool name + args, execute the tool
6. Inject OBSERVATION back as a user message
7. Loop

In [ ]:
def run_agent(user_query, max_iterations=10, verbose=True):
    """
    A ReAct agent from scratch.
    No frameworks. Just an LLM + tools + a loop.
    """
    # Build system prompt with tool descriptions
    tool_desc = "\n".join(f"- {t['desc']}" for t in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1}/{max_iterations} ---")

        # STEP 1: Ask the LLM what to do
        response = client.chat.completions.create(
            model=FREE_MODEL,
            messages=messages,
            temperature=0,
            max_tokens=800,
        )

        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})

        # STEP 2: Parse the response
        thought_match = re.search(
            r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)", text, re.DOTALL
        )
        thought = thought_match.group(1).strip() if thought_match else ""

        if verbose and thought:
            print(f"💭 THOUGHT: {thought[:200]}")

        # Check for FINAL_ANSWER
        if "FINAL_ANSWER:" in text:
            answer = text.split("FINAL_ANSWER:")[-1].strip()
            if verbose:
                print(f"\n{'='*60}")
                print(f"✅ AGENT FINISHED in {i+1} iteration(s)")
                print(f"{'='*60}")
                print(f"📝 ANSWER: {answer[:500]}")
            return {"answer": answer, "iterations": i + 1}

        # Check for ACTION
        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\n|$)", text, re.DOTALL)

        if not action_match:
            messages.append({
                "role": "user",
                "content": "Please respond with either ACTION + ACTION_INPUT or FINAL_ANSWER."
            })
            if verbose:
                print("⚠️  Format issue — nudging...")
            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        # STEP 3: Execute the tool
        if tool_name not in TOOLS:
            observation = json.dumps({
                "error": f"Unknown tool '{tool_name}'. Available: {list(TOOLS.keys())}"
            })
        else:
            try:
                if raw_input.startswith("{"):
                    args = json.loads(raw_input)
                else:
                    args = {"query": raw_input.strip("\"'")}
                observation = TOOLS[tool_name]["fn"](**args)
            except Exception as e:
                observation = json.dumps({"error": f"Failed to call {tool_name}: {e}"})

        if verbose:
            print(f"🔧 ACTION: {tool_name}({raw_input[:80]})")
            obs_preview = observation[:200] + "..." if len(observation) > 200 else observation
            print(f"👁️  OBSERVATION: {obs_preview}")

        # STEP 4: Feed observation back
        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

    if verbose:
        print(f"\n⚠️  Max iterations ({max_iterations}) reached.")
    return {"answer": "Max iterations reached.", "iterations": max_iterations}

### Test — Simple question

Watch the loop: `THOUGHT → ACTION → OBSERVATION → FINAL_ANSWER`

In [ ]:
result = run_agent("What is 123*123?")

In [ ]:
result = run_agent(
    "Search Wikipedia for 'Wakanda' and tell me about its history."
)

### Test — Multi-hop reasoning

Requires chaining multiple searches + calculation.

In [ ]:
result = run_agent(
    "Who was born first: the person who wrote 'Romeo and Juliet' "
    "or the person who painted the Mona Lisa? By how many years?"
)

---
## Part 4: Trace & Analyze the Agent

Let's instrument the agent to understand **costs, token growth, and behavior patterns**. This is the context engineering problem — every iteration re-sends everything.

In [ ]:
def run_agent_traced(user_query, max_iterations=10):
    """Same agent, but collects detailed trace data."""
    tool_desc = "\n".join(f"- {t['desc']}" for t in TOOLS.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    trace = []
    token_log = []

    for i in range(max_iterations):
        response = client.chat.completions.create(
            model=FREE_MODEL, messages=messages,
            temperature=0, max_tokens=800,
        )

        tokens_in = response.usage.prompt_tokens
        tokens_out = response.usage.completion_tokens
        token_log.append({"iter": i+1, "in": tokens_in, "out": tokens_out})

        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})

        thought_match = re.search(r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else ""

        if "FINAL_ANSWER:" in text:
            answer = text.split("FINAL_ANSWER:")[-1].strip()
            trace.append({"step": i+1, "type": "✅ FINAL", "thought": thought, "answer": answer})
            break

        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\n|$)", text, re.DOTALL)

        if not action_match:
            messages.append({"role": "user", "content": "Use ACTION or FINAL_ANSWER format."})
            trace.append({"step": i+1, "type": "⚠️ FORMAT", "thought": thought})
            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        if tool_name in TOOLS:
            try:
                args = json.loads(raw_input) if raw_input.startswith("{") else {"query": raw_input.strip("\"'")}
                observation = TOOLS[tool_name]["fn"](**args)
            except Exception as e:
                observation = json.dumps({"error": str(e)})
        else:
            observation = json.dumps({"error": f"Unknown tool: {tool_name}"})

        trace.append({
            "step": i+1, "type": "🔧 ACTION",
            "thought": thought[:150], "tool": tool_name,
            "input": raw_input[:100], "observation": observation[:200],
        })
        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

    return trace, token_log

In [ ]:
trace, tokens = run_agent_traced(
    "Who lived longer — Albert Einstein or Isaac Newton? By how many years?"
)

print("=" * 60)
print("AGENT TRACE")
print("=" * 60)
for step in trace:
    print(f"\nStep {step['step']} {step['type']}")
    print(f"  Thought: {step.get('thought', '')[:120]}")
    if step['type'] == '🔧 ACTION':
        print(f"  Tool:    {step['tool']}({step['input'][:60]})")
        print(f"  Result:  {step['observation'][:120]}")
    elif 'answer' in step:
        print(f"  Answer:  {step['answer'][:200]}")

print("\n" + "=" * 60)
print("TOKEN USAGE PER ITERATION")
print("=" * 60)
total = 0
for t in tokens:
    subtotal = t['in'] + t['out']
    total += subtotal
    bar = "█" * (t['in'] // 50)
    print(f"  Iter {t['iter']}: {t['in']:>5} in + {t['out']:>4} out = {subtotal:>5} │ {bar}")

print(f"\n  TOTAL: {total:,} tokens")
print(f"  → prompt_tokens GROWS each iteration — the context engineering problem.")

---
## Part 5: Level Up — Build a Code Agent

Give the agent **file I/O + Python execution**. These are essentially the same tools Claude Code kinda has:
- `read_file` → read a file
- `write_file` → write a file
- `run_python` → execute Python
- `list_files` → see directory contents

In [ ]:
def read_file(path):
    """Read a file's contents."""
    try:
        with open(path, "r") as f:
            content = f.read()
        return json.dumps({"path": path, "content": content[:3000], "truncated": len(content) > 3000})
    except Exception as e:
        return json.dumps({"error": str(e)})

def write_file(path, content):
    """Write content to a file."""
    try:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        with open(path, "w") as f:
            f.write(content)
        return json.dumps({"status": "success", "path": path, "bytes": len(content)})
    except Exception as e:
        return json.dumps({"error": str(e)})

def run_python(code):
    """Execute Python code and return stdout/stderr."""
    try:
        result = subprocess.run(
            ["python3", "-c", code],
            capture_output=True, text=True, timeout=15,
        )
        return json.dumps({
            "stdout": result.stdout[:2000] if result.stdout else "",
            "stderr": result.stderr[:2000] if result.stderr else "",
            "exit_code": result.returncode,
        })
    except subprocess.TimeoutExpired:
        return json.dumps({"error": "Timed out (15s limit)"})
    except Exception as e:
        return json.dumps({"error": str(e)})

def list_files(path="."):
    """List files in a directory."""
    try:
        items = sorted(os.listdir(path))
        return json.dumps({"path": path, "files": items[:50]})
    except Exception as e:
        return json.dumps({"error": str(e)})

CODE_TOOLS = {
    "read_file": {"fn": read_file, "desc": "read_file(path: str) — Read a file and return its contents."},
    "write_file": {"fn": write_file, "desc": 'write_file(path: str, content: str) — Write content to a file.'},
    "run_python": {"fn": run_python, "desc": "run_python(code: str) — Execute Python code. Returns stdout/stderr."},
    "list_files": {"fn": list_files, "desc": "list_files(path: str) — List files in a directory."},
    "calculate": {"fn": calculate_math, "desc": "calculate(expression: str) — Evaluate a math expression."},
}

print(f"Code tools: {list(CODE_TOOLS.keys())}")

In [ ]:
CODE_SYSTEM_PROMPT = """You are a coding agent. You write, run, and debug Python code to solve tasks.

Available tools:
{tool_descriptions}

Format — to use a tool:

THOUGHT: <your reasoning>
ACTION: <tool_name>
ACTION_INPUT: {{"arg1": "value1", "arg2": "value2"}}

Format — when done:

THOUGHT: <final reasoning>
FINAL_ANSWER: <your answer>

## Rules
- ONE action per turn. Wait for OBSERVATION.
- After writing code to a file, always run_python to TEST it.
- If a test fails, read the error, fix the code, try again.
- Verify your work before giving FINAL_ANSWER.
- Include print() statements so you can see output.
"""

def run_code_agent(user_query, max_iterations=15, verbose=True):
    """ReAct agent with coding tools."""
    tool_desc = "\n".join(f"- {t['desc']}" for t in CODE_TOOLS.values())
    system = CODE_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 TASK: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1}/{max_iterations} ---")

        response = client.chat.completions.create(
            model=FREE_MODEL, messages=messages,
            temperature=0, max_tokens=1500,
        )

        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})

        thought_match = re.search(r"THOUGHT:\s*(.+?)(?=ACTION:|FINAL_ANSWER:|$)", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else ""
        if verbose and thought:
            print(f"💭 THOUGHT: {thought[:200]}")

        if "FINAL_ANSWER:" in text:
            answer = text.split("FINAL_ANSWER:")[-1].strip()
            if verbose:
                print(f"\n{'='*60}")
                print(f"✅ DONE in {i+1} iteration(s)")
                print(f"{'='*60}")
                print(f"📝 {answer[:500]}")
            return answer

        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\nTHOUGHT|\nACTION|\nFINAL|$)", text, re.DOTALL)

        if not action_match:
            messages.append({"role": "user", "content": "Use ACTION + ACTION_INPUT or FINAL_ANSWER."})
            if verbose:
                print("⚠️  Format issue, nudging...")
            continue

        tool_name = action_match.group(1).strip()
        raw_input = input_match.group(1).strip() if input_match else "{}"

        if verbose:
            print(f"🔧 ACTION: {tool_name}")

        if tool_name not in CODE_TOOLS:
            observation = json.dumps({"error": f"Unknown tool '{tool_name}'. Available: {list(CODE_TOOLS.keys())}"})
        else:
            try:
                args = json.loads(raw_input)
                observation = CODE_TOOLS[tool_name]["fn"](**args)
            except json.JSONDecodeError:
                try:
                    observation = CODE_TOOLS[tool_name]["fn"](raw_input.strip("\"'"))
                except Exception as e:
                    observation = json.dumps({"error": f"Parse + fallback failed: {e}"})
            except Exception as e:
                observation = json.dumps({"error": str(e)})

        if verbose:
            obs_short = observation[:250] + "..." if len(observation) > 250 else observation
            print(f"👁️  OBSERVATION: {obs_short}")

        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

    return "Max iterations reached."

### Test — Write and test code

Watch the agent write code, run it, see output (or errors), and iterate.

In [ ]:
result = run_code_agent(
    "Write a Python function called 'is_prime(n)' that checks if a number is prime. "
    "Save it to 'prime.py'. Then write a test script that imports it and tests with: "
    "1, 2, 13, 15, 97, 100. Print the results."
)

In [ ]:
result = run_code_agent(
    "Create a CSV file called 'employees.csv' with columns: name, department, salary. "
    "Add at least 10 rows of realistic data across 3 departments (Engineering, Sales, Marketing). "
    "Then write a script that reads the CSV, calculates average salary per department, "
    "and prints a formatted report."
)

# Check the files
if os.path.exists("employees.csv"):
    print("\n--- employees.csv ---")
    with open("employees.csv") as f:
        print(f.read())

### Test — Algorithmic task

In [ ]:
result = run_code_agent(
    "Write a Python program using the Sieve of Eratosthenes to find all primes up to 1000. "
    "Save it as 'sieve.py'. Run it and print: the count of primes, the first 10, and the last 10."
)

---
## Part 6: Native Function-Calling Agent

Our Part 3 agent uses **string parsing** (regex on THOUGHT/ACTION). Modern APIs support **native function calling** — the model outputs structured `tool_calls` that don't need regex.

Let's rebuild using OpenRouter's tool calling API. This is closer to how production agents work.

In [ ]:
NATIVE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Search Wikipedia for information about a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Topic to search"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression like '(5+3)*2'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get the current date and time.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
]

NATIVE_FNS = {
    "search_wikipedia": search_wikipedia,
    "calculate": calculate_math,
    "get_current_date": get_current_date,
}


def run_native_agent(user_query, max_iterations=10, verbose=True):
    """
    Agent using native function calling.
    No regex — the API returns structured tool_calls.
    """
    messages = [
        {"role": "system", "content":
            "You are a helpful research assistant. Use tools to find information. "
            "Think step by step. After gathering enough info, give a complete answer."},
        {"role": "user", "content": user_query},
    ]

    if verbose:
        print(f"\n{'='*60}")
        print(f"🧑 USER: {user_query}")
        print(f"{'='*60}")

    for i in range(max_iterations):
        if verbose:
            print(f"\n--- Iteration {i+1} ---")

        response = client.chat.completions.create(
            model=TOOL_MODEL, messages=messages,
            tools=NATIVE_TOOLS, temperature=0, max_tokens=800,
        )

        msg = response.choices[0].message
        finish = response.choices[0].finish_reason

        # Text response = done
        if finish == "stop" and msg.content:
            if verbose:
                print(f"✅ FINAL ANSWER:\n  {msg.content[:500]}")
            return {"answer": msg.content, "iterations": i + 1}

        # Tool calls
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

                if verbose:
                    print(f"🔧 TOOL: {fn_name}({json.dumps(fn_args)[:80]})")

                if fn_name in NATIVE_FNS:
                    result = NATIVE_FNS[fn_name](**fn_args)
                else:
                    result = json.dumps({"error": f"Unknown tool: {fn_name}"})

                if verbose:
                    print(f"👁️  RESULT: {result[:150]}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result,
                })
        else:
            if msg.content:
                messages.append({"role": "assistant", "content": msg.content})
            else:
                break

    return {"answer": "Max iterations reached", "iterations": max_iterations}

In [ ]:
result = run_native_agent(
    "What's the date today and what will be tomorrow?"
)

Todos

🏋️ Todo 1: Context Management Our agent re-sends everything each iteration. Add summarization: after N iterations, compress the conversation history. Compare token usage before/after.

🏋️ Todo 2: Planning Step Modify the agent to first create a PLAN (list of steps), then execute each one. Compare Plan→Execute vs pure ReAct on the same task.

🏋️ Todo 3: Multi-Agent Create a 'researcher' (Wikipedia tools) and a 'coder' (file/code tools). Build an orchestrator that routes subtasks to the right agent.

🏋️ Todo 4: Web Browsing Add a fetch_webpage(url) tool. The agent can now follow links. What new failure modes appear?

## Todo 1: Context Management

Add a summarization step after every `summary_every` iterations. Compare the prompt-token growth of the original agent with the summarized version on the same question.

In [ ]:
def summarize_messages(messages):
    """Ask the model to compress older turns into one context message."""
    transcript = "\n".join(
        f"{message['role'].upper()}: {message.get('content', '')}"
        for message in messages[1:]
    )
    response = client.chat.completions.create(
        model=FREE_MODEL,
        messages=[{
            "role": "user",
            "content": (
                "Summarize this agent transcript. Keep the original task, facts, "
                "tool results, unfinished work, and constraints. Be concise.\n\n"
                + transcript
            ),
        }],
        temperature=0,
        max_tokens=500,
    )
    return response.choices[0].message.content or ""


def run_agent_with_summary(user_query, summary_every=3, max_iterations=10):
    """Run the ReAct agent while periodically compressing its context."""
    tool_desc = "\n".join(f"- {tool['desc']}" for tool in TOOLS.values())
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT.format(tool_descriptions=tool_desc)},
        {"role": "user", "content": user_query},
    ]
    token_log = []

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=FREE_MODEL,
            messages=messages,
            temperature=0,
            max_tokens=800,
        )
        token_log.append(response.usage.prompt_tokens)
        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})

        if "FINAL_ANSWER:" in text:
            return text.split("FINAL_ANSWER:", 1)[1].strip(), token_log

        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\n|$)", text, re.DOTALL)
        if not action_match:
            messages.append({"role": "user", "content": "Use ACTION + ACTION_INPUT or FINAL_ANSWER."})
            continue

        tool_name = action_match.group(1)
        raw_input = input_match.group(1).strip() if input_match else "{}"
        try:
            args = json.loads(raw_input) if raw_input.startswith("{") else {"query": raw_input.strip("\"'")}
            observation = TOOLS[tool_name]["fn"](**args)
        except Exception as error:
            observation = json.dumps({"error": str(error)})
        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

        if (iteration + 1) % summary_every == 0:
            summary = summarize_messages(messages)
            messages = [
                messages[0],
                {"role": "user", "content": f"CONDENSED CONTEXT:\n{summary}"},
            ]

    return "Max iterations reached.", token_log

question = "Who lived longer: Albert Einstein or Isaac Newton, and by how many years?"
_, original_tokens = run_agent_traced(question)
_, summarized_tokens = run_agent_with_summary(question, summary_every=2)
print("Original prompt tokens:", original_tokens)
print("Summarized prompt tokens:", summarized_tokens)
print("Original total:", sum(item["in"] for item in original_tokens))
print("Summarized total:", sum(summarized_tokens))

## Todo 2: Planning Step

Build a plan first, then execute one step at a time. Compare the planned agent with pure ReAct on the same multi-hop question.

In [ ]:
def create_plan(user_query):
    response = client.chat.completions.create(
        model=FREE_MODEL,
        messages=[
            {"role": "system", "content": "Return only a JSON list of short steps."},
            {"role": "user", "content": f"Create a plan to solve: {user_query}"},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw_plan = response.choices[0].message.content or "[]"
    try:
        return json.loads(raw_plan)
    except json.JSONDecodeError:
        return [line.strip(" -0123456789.") for line in raw_plan.splitlines() if line.strip()]


def run_planned_agent(user_query, max_steps=6):
    plan = create_plan(user_query)[:max_steps]
    completed = []
    for step in plan:
        result = run_agent(
            f"Solve this step: {step}\nOriginal task: {user_query}\n"
            f"Previous results: {completed}",
            max_iterations=3,
            verbose=False,
        )
        completed.append({"step": step, "result": result["answer"]})

    final_prompt = "Write the final answer using these completed steps:\n" + json.dumps(completed)
    response = client.chat.completions.create(
        model=FREE_MODEL,
        messages=[{"role": "user", "content": final_prompt}],
        temperature=0,
        max_tokens=500,
    )
    return {"plan": plan, "answer": response.choices[0].message.content or "", "steps": completed}

planned_question = "Who lived longer: Albert Einstein or Isaac Newton, and by how many years?"
planned_result = run_planned_agent(planned_question)
react_result = run_agent(planned_question, verbose=False)
print("PLAN:")
for number, step in enumerate(planned_result["plan"], 1):
    print(f"  {number}. {step}")
print("\nPLAN → EXECUTE ANSWER:", planned_result["answer"])
print("\nPURE REACT ANSWER:", react_result["answer"])
print("\nComparison: planning makes the intended sequence explicit; ReAct adapts after each observation.")

## Todo 3: Multi-Agent Orchestrator

Create a researcher with Wikipedia tools and a coder with file/code tools. Route each subtask to the specialist that can actually perform it, then combine their outputs.

In [ ]:
import subprocess

RESEARCHER_TOOLS = {
    "search_wikipedia": TOOLS["search_wikipedia"],
    "calculate": TOOLS["calculate"],
    "get_current_date": TOOLS["get_current_date"],
}
CODER_TOOLS = CODE_TOOLS


def run_specialist(user_query, tools, max_iterations=8):
    """Run a compact ReAct specialist with its own tool registry."""
    descriptions = "\n".join(f"- {tool['desc']}" for tool in tools.values())
    system = REACT_SYSTEM_PROMPT.format(tool_descriptions=descriptions)
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_query},
    ]
    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=FREE_MODEL,
            messages=messages,
            temperature=0,
            max_tokens=900,
        )
        text = response.choices[0].message.content or ""
        messages.append({"role": "assistant", "content": text})
        if "FINAL_ANSWER:" in text:
            return text.split("FINAL_ANSWER:", 1)[1].strip()

        action_match = re.search(r"ACTION:\s*(\w+)", text)
        input_match = re.search(r"ACTION_INPUT:\s*(.+?)(?:\n|$)", text, re.DOTALL)
        if not action_match:
            messages.append({"role": "user", "content": "Use ACTION + ACTION_INPUT or FINAL_ANSWER."})
            continue

        tool_name = action_match.group(1)
        raw_input = input_match.group(1).strip() if input_match else "{}"
        try:
            args = json.loads(raw_input)
            observation = tools[tool_name]["fn"](**args)
        except Exception as error:
            observation = json.dumps({"error": str(error)})
        messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
    return "Specialist reached its iteration limit."


def orchestrate(task):
    """Route research and implementation subtasks to different specialists."""
    research = run_specialist(
        "Research the facts needed for this task and return a concise brief:\n" + task,
        RESEARCHER_TOOLS,
    )
    coding = run_specialist(
        "Use the research brief below to implement and test the requested code.\n"
        f"Task: {task}\nResearch brief: {research}",
        CODER_TOOLS,
    )
    return {"research": research, "coding": coding}

multi_agent_result = orchestrate(
    "Find when the Sieve of Eratosthenes was developed, then write sieve_report.py "
    "that prints the number of primes up to 100 and the historical fact."
)
print("RESEARCHER:\n", multi_agent_result["research"])
print("\nCODER:\n", multi_agent_result["coding"])
print("\nThe orchestrator separates permissions and prompts: the researcher cannot write files, and the coder need not browse Wikipedia.")

## Todo 4: Web Browsing

Add a `fetch_webpage(url)` tool and let the agent follow a page. Observe failure modes such as timeouts, redirects, HTML noise, paywalls, robots restrictions, and prompt injection in page content.

In [ ]:
from urllib.parse import urlparse


def fetch_webpage(url):
    """Fetch a bounded text snapshot while making common failures explicit."""
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        return json.dumps({"error": "Only absolute http:// or https:// URLs are allowed."})
    try:
        response = requests.get(
            url,
            headers={"User-Agent": "class-agent-demo/1.0"},
            timeout=10,
            allow_redirects=True,
        )
        content_type = response.headers.get("content-type", "")
        if response.status_code >= 400:
            return json.dumps({"error": f"HTTP {response.status_code}", "url": response.url})
        if "text" not in content_type and "json" not in content_type:
            return json.dumps({"error": f"Unsupported content type: {content_type}"})
        text = response.text[:6000]
        return json.dumps({
            "url": response.url,
            "status": response.status_code,
            "content_type": content_type,
            "text": text,
            "truncated": len(response.text) > 6000,
        })
    except requests.Timeout:
        return json.dumps({"error": "Request timed out."})
    except requests.RequestException as error:
        return json.dumps({"error": str(error)})


BROWSER_TOOLS = {
    "fetch_webpage": {
        "fn": fetch_webpage,
        "desc": "fetch_webpage(url: str) — Fetch a bounded text snapshot from an HTTP(S) URL.",
    },
    "search_wikipedia": TOOLS["search_wikipedia"],
}

browser_result = run_specialist(
    "Open https://en.wikipedia.org/wiki/Agent-based_model and summarize the page in three bullets.",
    BROWSER_TOOLS,
)
print(browser_result)
print("\nFailure modes to investigate:")
for failure in [
    "timeouts and transient network errors",
    "redirects, non-text content, and very large pages",
    "HTTP errors, paywalls, robots restrictions, and stale content",
    "untrusted page text attempting to override the agent's instructions",
]:
    print("-", failure)